# Análise Comparativa — Corte de Barras

**Disciplina:** Análise de Algoritmos — IFCE

**Tema 7.1:** Corte de Barras (Rod Cutting)

**Autor:** [Seu Nome]

---

## 1. Introdução

O problema de **Corte de Barras** consiste em: dada uma barra de comprimento $n$ e uma tabela de preços $p[i]$ para cada pedaço de tamanho $i$ ($1 \leq i \leq n$), encontrar a melhor forma de cortar a barra para maximizar a receita total.

Neste experimento, implementamos e comparamos três algoritmos:
- **Guloso ingênuo**: escolhe localmente o pedaço com melhor relação preço/tamanho
- **Força Bruta**: testa recursivamente todas as partições possíveis ($O(2^n)$)
- **Programação Dinâmica (memoização)**: solução recursiva com memoização ($O(n^2)$)

## 2. Implementação

Os algoritmos estão implementados em `src/algorithms.py`.

### 2.1 Algoritmo Guloso

```python
def corte_guloso(n, precos):
    restante = n
    cortes = []
    while restante > 0:
        melhor_razao = -1.0
        melhor_tam = 1
        for t in range(1, restante + 1):
            razao = precos[t - 1] / t
            if razao > melhor_razao:
                melhor_razao = razao
                melhor_tam = t
        cortes.append(melhor_tam)
        restante -= melhor_tam
    return sum(precos[t-1] for t in cortes), cortes
```

### 2.2 Força Bruta

```python
def corte_forca_bruta(n, precos):
    if n == 0:
        return 0, []
    melhor_valor = -inf
    melhor_cortes = []
    for i in range(1, n + 1):
        v_resto, c_resto = corte_forca_bruta(n - i, precos)
        valor = precos[i - 1] + v_resto
        if valor > melhor_valor:
            melhor_valor = valor
            melhor_cortes = [i] + c_resto
    return melhor_valor, melhor_cortes
```

### 2.3 Programação Dinâmica (Memoização)

```python
def corte_pd_memo(n, precos):
    memo_valor = {}
    memo_corte = {}
    
    def f(m):
        if m == 0: return 0
        if m in memo_valor: return memo_valor[m]
        melhor = -inf
        for i in range(1, m + 1):
            valor = precos[i-1] + f(m - i)
            if valor > melhor:
                melhor = valor
                melhor_primeiro = i
        memo_valor[m] = melhor
        memo_corte[m] = melhor_primeiro
        return melhor
    
    valor_total = f(n)
    # Reconstrói os cortes
    cortes = []
    resto = n
    while resto > 0:
        c = memo_corte[resto]
        cortes.append(c)
        resto -= c
    return valor_total, cortes
```

## 3. Verificação de Corretude

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from src.verify import (
    verificar_corretude_fb_vs_pd,
    verificar_guloso_vs_pd,
    exemplo_guloso_erra
)

# Verifica FB vs PD nos tamanhos pequenos
tamanhos_pequenos = list(range(5, 22, 2))
fb_ok = verificar_corretude_fb_vs_pd(tamanhos_pequenos)
print(f"\n>>> FB vs PD: {'TODOS OK' if fb_ok else 'COM ERROS'}\n")

# Verifica Guloso vs PD em todos os tamanhos
tamanhos_todos = list(range(5, 101, 5))
diffs = verificar_guloso_vs_pd(tamanhos_todos)

### 3.1 Exemplo onde o Guloso Erra

In [ ]:
precos, v_gul, c_gul, v_pd, c_pd = exemplo_guloso_erra()

**Análise:** Nesta tabela de preços, o algoritmo guloso escolhe o pedaço com melhor relação preço/tamanho em cada passo, mas essa escolha local leva a um resultado sub-ótimo. A PD, ao considerar todas as combinações via memoização, encontra o valor ótimo.

Por exemplo, se o guloso escolhe um pedaço de tamanho 3 (R\$ 8,00, razão 2.67) em vez de dois pedaços de tamanho 2 (R\$ 5,00 + R\$ 5,00 = R\$ 10,00, razão 2.50 cada), ele pode obter um valor total menor que o ótimo.

## 4. Benchmark de Tempo

In [ ]:
from src.benchmark import rodar_benchmark, gerar_graficos
df = rodar_benchmark()
grafico_path = gerar_graficos(df)

### 4.1 Visualização dos Resultados

In [ ]:
from IPython.display import Image, display
display(Image(filename=grafico_path))

## 5. Discussão dos Resultados

### 5.1 Complexidade Teórica vs Observada

| Algoritmo | Complexidade Teórica | Comportamento Observado |
|-----------|---------------------|------------------------|
| Guloso | O(n²) no pior caso | Linear na prática (cada passo percorre O(n) e reduz n) |
| Força Bruta | O(2ⁿ) | Crescimento exponencial — inviável para n > 25 |
| PD (Memo) | O(n²) | Quadrático — escala bem até n ≈ 500+ |

### 5.2 Inviabilidade da Força Bruta

A Força Bruta testa **todas as $2^{n-1}$ partições possíveis** da barra. Para $n=20$, isso já significa ~500 mil combinações. Para $n=30$, mais de 500 milhões. Nos experimentos, a Força Bruta tornou-se impraticável (tempo > 1 minuto) a partir de $n \approx 25$.

### 5.3 Escalabilidade da PD

A Programação Dinâmica com memoização resolve o problema em $O(n^2)$ — para cada subproblema $m$ (de 1 a $n$), testa $m$ possibilidades de primeiro corte. Nos testes, a PD executou em menos de 0.1s para $n=500$, confirmando que é a abordagem recomendada para este problema.

### 5.4 Qualidade da Solução Gulosa

O Guloso é o mais rápido, mas **não garante o ótimo**. A diferença entre o valor da PD e do Guloso variou ao longo dos experimentos (ver gráfico 3). Em alguns casos específicos (como o exemplo da seção 3.1), a diferença pode ser significativa.

### 5.5 Conclusão

Para o problema de Corte de Barras, a **Programação Dinâmica** é a melhor escolha: oferece solução ótima com complexidade polinomial $O(n^2)$, escalável para entradas grandes. O Guloso pode ser usado quando velocidade é crítica e uma solução aproximada é aceitável. A Força Bruta é apenas um conceito didático — inviável na prática para qualquer $n > 25$.